In [1]:
!pip install -q transformers==4.46.3
!pip install -q datasets==3.1.0
!pip install -q accelerate==1.1.1
!pip install -q sentence-transformers
!pip install -q evaluate
!pip install -q scikit-learn
!pip install -q imbalanced-learn
!pip install -q transformers accelerate sentencepiece
!pip uninstall -y torchao
!pip install -U torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 18.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.9.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully unins

In [3]:
from google.colab import drive
drive.mount('/content/gdrive')

import os
os.chdir('/content/gdrive/My Drive/MARS')
print("✅ Working directory:", os.getcwd())

Mounted at /content/gdrive
✅ Working directory: /content/gdrive/My Drive/MARS


In [4]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [5]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

In [6]:
df = pd.read_csv("customer_support_tickets.csv")
print("Shape:", df.shape)
df.head()

Shape: (20000, 12)


,Ticket_ID,Customer_Name,Customer_Email,Ticket_Subject,Ticket_Description,Issue_Category,Priority_Level,Ticket_Channel,Submission_Date,Resolution_Time_Hours,Assigned_Agent,Satisfaction_Score
0,TKT-100000,George Simon,lisastrickland@example.com,Hours of operation - Individual,"Hi Support, Where is your headquarters located...",General Inquiry,High,Web Form,2025-07-02,43,David Kim,5
1,TKT-100001,Scott Thompson,wevans@example.org,Data not syncing - Card,"Hi Support, The application crashes every time...",Technical,High,Chat,2025-06-28,41,Elena Rodriguez,5
2,TKT-100002,Jennifer Smith,oleonard@example.net,2FA issues - Question,"Hi Support, How do I upgrade to the Enterprise...",Account,High,Web Form,2025-02-05,7,Anya Sharma,5
3,TKT-100003,Rachel Bullock,katherine67@example.net,Login failed - Let,"Hi Support, The dashboard is not loading any d...",Technical,Low,Web Form,2025-03-20,41,Anya Sharma,5
4,TKT-100004,Thomas Parks DDS,raykelsey@example.com,Refund status - Attention,"Hi Support, I have been trying to update my pa...",Billing,Medium,Email,2025-04-27,40,David Kim,5


# Preprocessing

In [9]:
required_cols = [
    "Ticket_ID",
    "Ticket_Subject",
    "Ticket_Description",
    "Priority_Level",
    "Ticket_Channel",
    "Issue_Category",
    "Resolution_Time_Hours",
    "Customer_Email"
]

df = df[required_cols].copy()
print(df.shape)

(20000, 8)


In [10]:
def clean_ticket_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    # Remove Hi Support
    text = re.sub(r"^Hi Support,\s*","", text,flags=re.IGNORECASE)
    # Keep only first sentence
    text = re.split(r"[?.!]", text)[0]
    return text.strip()

In [11]:
df["clean_description"] = (df["Ticket_Description"].apply(clean_ticket_text))

In [12]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+","",text)
    text = re.sub(r"[^a-zA-Z0-9 ]"," ",text)
    text = re.sub(r"\s+"," ",text)
    return text.strip()

In [13]:
df["clean_text"] = (df["Ticket_Subject"].fillna("")+ " "+ df["clean_description"].fillna(""))

In [14]:
df["clean_text"] = (df["clean_text"].apply(clean_text))

In [15]:
df[["Ticket_Subject", "clean_description", "clean_text"]].head()

,Ticket_Subject,clean_description,clean_text
0,Hours of operation - Individual,Where is your headquarters located,hours of operation individual where is your he...
1,Data not syncing - Card,The application crashes every time I open the ...,data not syncing card the application crashes ...
2,2FA issues - Question,How do I upgrade to the Enterprise plan,2fa issues question how do i upgrade to the en...
3,Login failed - Let,"The dashboard is not loading any data, just a ...",login failed let the dashboard is not loading ...
4,Refund status - Attention,I have been trying to update my payment method...,refund status attention i have been trying to ...


# Pseudo-labels

In [16]:
critical_words = [
    "outage",
    "security",
    "fraud",
    "breach",
    "data loss",
    "stolen card",
    "unauthorized",
    "cannot login",
    "system down",
    "payment failed",
    "account hacked",
    "service unavailable",
    "data corruption"
]

high_words = [
    "crash",
    "error",
    "failed",
    "sync",
    "invoice discrepancy",
    "login issue",
    "payment issue",
    "screen freezes",
    "api error",
    "application crash",
    "data not syncing"
]

In [17]:
def rule_score(text):
    score = 0
    evidence = []

    for word in critical_words:
        if word in text:
            score += 3
            evidence.append(word)

    for word in high_words:
        if word in text:
            score += 2
            evidence.append(word)

    if score >= 6:
        severity = 3

    elif score >= 4:
        severity = 2

    elif score >= 2:
        severity = 1

    else:
        severity = 0

    return severity, ",".join(evidence)

In [18]:
df[["rule_score", "rule_evidence"]] = (df["clean_text"].apply(lambda x: pd.Series(rule_score(x))))
df[["rule_score","rule_evidence"]].head()

,rule_score,rule_evidence
0,0,
1,3,"crash,sync,application crash,data not syncing"
2,0,
3,1,failed
4,0,


In [19]:
from sklearn.ensemble import RandomForestRegressor

priority_map={
    "Low":0,
    "Medium":1,
    "High":2,
    "Critical":3
}

df["priority_num"]=df["Priority_Level"].map(priority_map)

rf=RandomForestRegressor(
    n_estimators=200,
    max_depth=5,
    random_state=42
)

rf.fit(
    df[["Resolution_Time_Hours"]],
    df["priority_num"]
)

pred=rf.predict(
    df[["Resolution_Time_Hours"]]
)

df["resolution_score"]=(
    pred
    .clip(0,3)
    .round()
    .astype(int)
)

print(df["resolution_score"].value_counts())

resolution_score
1    20000
Name: count, dtype: int64


In [20]:

print(df.shape)
print(df["rule_score"].value_counts())
print(df["resolution_score"].value_counts())

(20000, 14)
rule_score
0    12563
2     3488
1     2988
3      961
Name: count, dtype: int64
resolution_score
1    20000
Name: count, dtype: int64


Generate Embeddings

In [21]:
c

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

K Means Clustering

In [22]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

silhouette_scores = {}

for k in range(2,9):

    kmeans_temp = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans_temp.fit_predict(
        embeddings
    )

    score = silhouette_score(
        embeddings,
        labels,
        sample_size=min(
            2000,
            len(df)
        ),
        random_state=42
    )

    silhouette_scores[k] = score

    print(
        f"k={k} -> {score:.4f}"
    )

best_k = max(
    silhouette_scores,
    key=silhouette_scores.get
)

print(
    f"\nBest k = {best_k}"
)

k=2 -> 0.0983
k=3 -> 0.1290
k=4 -> 0.1488
k=5 -> 0.1528
k=6 -> 0.1677
k=7 -> 0.1635
k=8 -> 0.1563

Best k = 6


In [23]:
kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(
    embeddings
)

print(
    df["cluster"]
    .value_counts()
    .sort_index()
)

cluster
0    1961
1    5764
2    3929
3    5166
4    2044
5    1136
Name: count, dtype: int64


In [24]:
for c in sorted(
    df["cluster"].unique()
):

    print(
        f"\nCLUSTER {c}"
    )

    print(
        df[
            df["cluster"] == c
        ]["Ticket_Subject"]
        .sample(
            min(
                20,
                len(
                    df[
                        df["cluster"] == c
                    ]
                )
            ),
            random_state=42
        )
        .tolist()
    )


CLUSTER 0
['Data not syncing - Enough', 'Installation issue - Create', 'Screen freezes - Actually', 'Data not syncing - Part', 'Login failed - Tonight', 'App crashing - Available', 'Data not syncing - Partner', 'Installation issue - Table', 'Screen freezes - Century', 'API Error 500 - Wear', 'Data not syncing - Minute', 'API Error 500 - Because', 'Data not syncing - Find', 'App crashing - Director', 'Installation issue - Reveal', 'App crashing - Political', 'Screen freezes - Seem', 'API Error 500 - Firm', 'Screen freezes - Front', 'Installation issue - Work']

CLUSTER 1
['Login failed - Nice', 'Data not syncing - Yes', '2FA issues - Second', 'Password reset - Usually', 'Subscription upgrade - Least', 'Password reset - Treatment', 'Login failed - Its', '2FA issues - Small', 'Change email - Hard', 'Data not syncing - Pay', 'Delete account - Both', 'Delete account - President', 'Delete account - Senior', 'Subscription upgrade - Example', 'Change email - Issue', 'Password reset - Bit', 'D

In [25]:
cluster_avg_resolution = (
    df.groupby("cluster")
      ["Resolution_Time_Hours"]
      .mean()
      .sort_values()
)

print(
    "\nCluster Avg Resolution:"
)

print(
    cluster_avg_resolution
)


Cluster Avg Resolution:
cluster
0    34.141254
5    35.970951
4    36.141879
1    37.296149
3    42.208285
2    43.241283
Name: Resolution_Time_Hours, dtype: float64


In [26]:
cluster_score_map = {

    cluster: rank

    for rank, cluster

    in enumerate(
        cluster_avg_resolution.index
    )
}

print(
    cluster_score_map
)

df["cluster_score"] = (
    df["cluster"]
      .map(cluster_score_map)
)

{0: 0, 5: 1, 4: 2, 1: 3, 3: 4, 2: 5}


In [27]:
print(
    "\nCluster Score Distribution:"
)

print(
    df["cluster_score"]
      .value_counts()
      .sort_index()
)

df[
    [
        "cluster",
        "cluster_score",
        "Resolution_Time_Hours"
    ]
].head()


Cluster Score Distribution:
cluster_score
0    1961
1    1136
2    2044
3    5764
4    5166
5    3929
Name: count, dtype: int64


,cluster,cluster_score,Resolution_Time_Hours
0,2,5,43
1,4,2,41
2,1,3,7
3,0,0,41
4,3,4,40


In [28]:
df.head()

,Ticket_ID,Ticket_Subject,Ticket_Description,Priority_Level,Ticket_Channel,Issue_Category,Resolution_Time_Hours,Customer_Email,clean_description,clean_text,rule_score,rule_evidence,priority_num,resolution_score,cluster,cluster_score
0,TKT-100000,Hours of operation - Individual,"Hi Support, Where is your headquarters located...",High,Web Form,General Inquiry,43,lisastrickland@example.com,Where is your headquarters located,hours of operation individual where is your he...,0,,2,1,2,5
1,TKT-100001,Data not syncing - Card,"Hi Support, The application crashes every time...",High,Chat,Technical,41,wevans@example.org,The application crashes every time I open the ...,data not syncing card the application crashes ...,3,"crash,sync,application crash,data not syncing",2,1,4,2
2,TKT-100002,2FA issues - Question,"Hi Support, How do I upgrade to the Enterprise...",High,Web Form,Account,7,oleonard@example.net,How do I upgrade to the Enterprise plan,2fa issues question how do i upgrade to the en...,0,,2,1,1,3
3,TKT-100003,Login failed - Let,"Hi Support, The dashboard is not loading any d...",Low,Web Form,Technical,41,katherine67@example.net,"The dashboard is not loading any data, just a ...",login failed let the dashboard is not loading ...,1,failed,0,1,0,0
4,TKT-100004,Refund status - Attention,"Hi Support, I have been trying to update my pa...",Medium,Email,Billing,40,raykelsey@example.com,I have been trying to update my payment method...,refund status attention i have been trying to ...,0,,1,1,3,4


In [29]:
df.to_csv("feature_engineered.csv",index=False)
print("Saved feature_engineered.csv")

Saved feature_engineered.csv


# LLM Severity training

In [ ]:
import pandas as pd

df = pd.read_csv("feature_engineered.csv")
print(df.shape)

(20000, 15)


In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
import torch

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
print("Model Loaded")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model Loaded


In [ ]:
def build_prompt(subject, description, category, channel):

    return f"""
You are a support ticket severity auditor.

Assign a severity score.

Severity scale:

0 = Low
1 = Medium
2 = High
3 = Critical

Definitions:

3:
Security incidents,
fraud,
account compromise,
service outage,
major operational disruption.

2:
Application crashes,
API failures,
login failures,
payment failures,
data synchronization failures.

1:
User-impacting issues with workarounds.

0:
Questions,
feature requests,
account updates,
minor requests.

Ticket Category:
{category}

Ticket Channel:
{channel}

Subject:
{subject}

Description:
{description}

Return ONLY ONE NUMBER.

Severity Score:
"""

In [ ]:
import re

def predict_severity( subject, description, category, channel):

    prompt = build_prompt(subject, description, category, channel)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=5, do_sample=False, temperature=0.0, pad_token_id=tokenizer.eos_token_id)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    numbers = re.findall(r"\b([0-3])\b", response)

    if len(numbers) > 0:
        return int(numbers[0])

    return None

Test On 20 tickets

In [ ]:
sample_df = df.sample(20, random_state=42)

for _, row in sample_df.iterrows():

    score = predict_severity(
        row["Ticket_Subject"],
        row["clean_description"],
        row["Issue_Category"],
        row["Ticket_Channel"]
    )

    print(row["Ticket_Subject"], "->",score)

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Change email - Trial -> 1
API Error 500 - Style -> 2
Login failed - Speak -> 2
Payment failed - Them -> 0
Login failed - Learn -> 1
Refund status - Strong -> 1
App crashing - Computer -> 2
Update credit card - Own -> 1
Data not syncing - Travel -> 1
Screen freezes - South -> 2
Stolen card - Political -> 3
Office location - Although -> 0
Payment failed - Guy -> 0
Change email - Position -> 0
App crashing - Common -> 2
Product question - Result -> 0
Password reset - Whom -> 0
Invoice discrepancy - Purpose -> 1
Change email - Blood -> 1
Unrecognized login - Southern -> 3


In [ ]:
import os
from tqdm import tqdm

CHECKPOINT_FILE = "mistral_scores_checkpoint.csv"
FINAL_FILE = "mistral_scores.csv"
SAVE_EVERY = 500

In [ ]:
if os.path.exists(CHECKPOINT_FILE):

    checkpoint = pd.read_csv(
        CHECKPOINT_FILE
    )

    checkpoint = checkpoint.drop_duplicates(
        subset="Ticket_ID",
        keep="last"
    )

    completed_ids = set(
        checkpoint["Ticket_ID"]
    )

    remaining_df = df[
        ~df["Ticket_ID"].isin(
            completed_ids
        )
    ].copy()

    results = checkpoint.to_dict(
        "records"
    )

    print(
        f"Completed: {len(checkpoint)}"
    )

    print(
        f"Remaining: {len(remaining_df)}"
    )

else:

    remaining_df = df.copy()

    results = []

    print(
        f"Starting fresh: {len(df)} tickets"
    )

Starting fresh: 20000 tickets


In [ ]:
counter = 0

for _, row in tqdm(
    remaining_df.iterrows(),
    total=len(remaining_df)
):

    try:

        score = predict_severity(

            row["Ticket_Subject"],

            row["clean_description"],

            row["Issue_Category"],

            row["Ticket_Channel"]

        )

    except Exception as e:

        print(
            "Error:",
            row["Ticket_ID"]
        )

        score = None

    results.append({

        "Ticket_ID":
        row["Ticket_ID"],

        "llm_score":
        score

    })

    counter += 1

    if counter % SAVE_EVERY == 0:

        checkpoint_df = (
            pd.DataFrame(results)
            .drop_duplicates(
                subset="Ticket_ID",
                keep="last"
            )
        )

        checkpoint_df.to_csv(
            CHECKPOINT_FILE,
            index=False
        )

        print(
            f"Checkpoint saved: {len(checkpoint_df)}"
        )

  2%|▎         | 500/20000 [06:48<4:24:08,  1.23it/s]

Checkpoint saved: 500


  5%|▌         | 1000/20000 [13:41<4:23:42,  1.20it/s]

Checkpoint saved: 1000


  8%|▊         | 1500/20000 [20:50<4:50:11,  1.06it/s]

...

Checkpoint saved: 19000


 98%|█████████▊| 19500/20000 [4:29:38<06:46,  1.23it/s]

Checkpoint saved: 19500


100%|██████████| 20000/20000 [4:36:25<00:00,  1.21it/s]

Checkpoint saved: 20000


In [ ]:
final_df = (
    pd.DataFrame(results)
    .drop_duplicates(
        subset="Ticket_ID",
        keep="last"
    )
)

final_df.to_csv(
    FINAL_FILE,
    index=False
)

print("\nSaved:", FINAL_FILE)

print(
    "\nRows:",
    len(final_df)
)

print(
    "Unique IDs:",
    final_df["Ticket_ID"]
    .nunique()
)


Saved: mistral_scores.csv

Rows: 20000
Unique IDs: 20000


In [ ]:
print(
    final_df["llm_score"]
    .value_counts(
        dropna=False
    )
)

llm_score
0    8677
1    7049
2    3054
3    1220
Name: count, dtype: int64


In [ ]:
print(
    final_df.shape
)

print(
    final_df["Ticket_ID"]
    .duplicated()
    .sum()
)

(20000, 2)
0


# Fusion

In [30]:
import pandas as pd

features = pd.read_csv(
    "feature_engineered.csv"
)

llm = pd.read_csv("mistral_scores.csv")

print(features.shape)
print(llm.shape)

(20000, 16)
(20000, 2)


In [31]:
df = features.merge(llm,on="Ticket_ID",how="left")

In [32]:
print(df.columns.tolist())

['Ticket_ID', 'Ticket_Subject', 'Ticket_Description', 'Priority_Level', 'Ticket_Channel', 'Issue_Category', 'Resolution_Time_Hours', 'Customer_Email', 'clean_description', 'clean_text', 'rule_score', 'rule_evidence', 'priority_num', 'resolution_score', 'cluster', 'cluster_score', 'llm_score']


In [33]:
df["severity_fusion"] = (
      0.40 * df["llm_score"]
    + 0.30 * df["cluster_score"]
    + 0.20 * df["resolution_score"]
    + 0.10 * df["rule_score"]
)

In [34]:
df["inferred_severity"] = 0

df.loc[df["severity_fusion"] >= 0.75, "inferred_severity"] = 1
df.loc[df["severity_fusion"] >= 1.50, "inferred_severity"] = 2
df.loc[df["severity_fusion"] >= 2.25, "inferred_severity"] = 3

In [35]:
print(
    df["inferred_severity"]
    .value_counts()
)

inferred_severity
2    10706
1     7341
3     1220
0      733
Name: count, dtype: int64


In [36]:
priority_map = {

    "Low": 0,
    "Medium": 1,
    "High": 2,
    "Critical": 3
}

In [37]:
df["assigned_priority_num"] = (
    df["Priority_Level"]
    .map(priority_map)
)

In [38]:
df[
    [
        "Priority_Level",
        "assigned_priority_num"
    ]
].head()

,Priority_Level,assigned_priority_num
0,High,2
1,High,2
2,High,2
3,Low,0
4,Medium,1


In [39]:
df["severity_delta"] = (
    df["inferred_severity"]
    - df["assigned_priority_num"]
)

In [40]:
df["severity_delta"].describe()

,severity_delta
count,20000.000000
mean,0.705850
std,1.058339
min,-3.000000
25%,0.000000
50%,1.000000
75%,1.000000
max,3.000000


In [41]:
df["mismatch"] = (
    abs(df["severity_delta"]) >= 2
).astype(int)

Check mismatch before training

In [42]:
print(df["mismatch"].value_counts())

print(
    df["mismatch"]
    .value_counts(normalize=True)
)

mismatch
0    14506
1     5494
Name: count, dtype: int64
mismatch
0    0.7253
1    0.2747
Name: proportion, dtype: float64


In [43]:
df["mismatch_type"] = "Consistent"

df.loc[
    df["severity_delta"] >= 2,
    "mismatch_type"
] = "Hidden Crisis"

df.loc[
    df["severity_delta"] <= -2,
    "mismatch_type"
] = "False Alarm"

In [44]:
print(
    df["mismatch"]
    .value_counts()
)

print(
    df["mismatch"]
    .value_counts(normalize=True)
)

mismatch
0    14506
1     5494
Name: count, dtype: int64
mismatch
0    0.7253
1    0.2747
Name: proportion, dtype: float64


In [45]:
print(df["inferred_severity"].value_counts())

print(df["mismatch"].value_counts())

print(df["mismatch"].value_counts(normalize=True))

print(
    pd.crosstab(
        df["Priority_Level"],
        df["mismatch"]
    )
)

print(
    df["mismatch_type"]
    .value_counts()
)

inferred_severity
2    10706
1     7341
3     1220
0      733
Name: count, dtype: int64
mismatch
0    14506
1     5494
Name: count, dtype: int64
mismatch
0    0.7253
1    0.2747
Name: proportion, dtype: float64
mismatch           0     1
Priority_Level            
Critical         880   418
High            3195   221
Low             2973  4743
Medium          7458   112
mismatch_type
Consistent       14506
Hidden Crisis     4855
False Alarm        639
Name: count, dtype: int64


In [46]:
df.to_csv(
    "pseudo_labeled_dataset.csv",
    index=False
)

print("Saved pseudo_labeled_dataset.csv")

Saved pseudo_labeled_dataset.csv


# Prepare dataset

In [47]:
df["text"] = (
    df["clean_text"]
    + " [CHANNEL] "
    + df["Ticket_Channel"].astype(str)
    + " [CATEGORY] "
    + df["Issue_Category"].astype(str)
)

In [48]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["mismatch"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["mismatch"],
    random_state=42
)

In [49]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

train_df["resolution_time_norm"] = scaler.fit_transform(
    train_df[["Resolution_Time_Hours"]]
)

val_df["resolution_time_norm"] = scaler.transform(
    val_df[["Resolution_Time_Hours"]]
)

test_df["resolution_time_norm"] = scaler.transform(
    test_df[["Resolution_Time_Hours"]]
)

print("\nTrain:")
print(
    train_df["resolution_time_norm"]
    .describe()
)

print("\nValidation:")
print(
    val_df["resolution_time_norm"]
    .describe()
)

print("\nTest:")
print(
    test_df["resolution_time_norm"]
    .describe()
)


Train:
count    14000.000000
mean         0.319632
std          0.295560
min          0.000000
25%          0.084034
50%          0.218487
75%          0.478992
max          1.000000
Name: resolution_time_norm, dtype: float64

Validation:
count    3000.000000
mean        0.321902
std         0.297390
min         0.000000
25%         0.084034
50%         0.218487
75%         0.478992
max         1.000000
Name: resolution_time_norm, dtype: float64

Test:
count    3000.000000
mean        0.328235
std         0.296535
min         0.000000
25%         0.092437
50%         0.226891
75%         0.487395
max         1.000000
Name: resolution_time_norm, dtype: float64


In [50]:
from sklearn.preprocessing import LabelEncoder

le_channel = LabelEncoder()

train_df["channel_encoded"] = le_channel.fit_transform(
    train_df["Ticket_Channel"].astype(str)
)

val_df["channel_encoded"] = le_channel.transform(
    val_df["Ticket_Channel"].astype(str)
)

test_df["channel_encoded"] = le_channel.transform(
    test_df["Ticket_Channel"].astype(str)
)

le_category = LabelEncoder()

train_df["category_encoded"] = le_category.fit_transform(
    train_df["Issue_Category"].astype(str)
)

val_df["category_encoded"] = le_category.transform(
    val_df["Issue_Category"].astype(str)
)

test_df["category_encoded"] = le_category.transform(
    test_df["Issue_Category"].astype(str)
)

In [51]:
print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(14000, 27)
(3000, 27)
(3000, 27)


In [52]:
print(train_df["mismatch"].value_counts(normalize=True))
print(val_df["mismatch"].value_counts(normalize=True))
print(test_df["mismatch"].value_counts(normalize=True))

mismatch
0    0.725286
1    0.274714
Name: proportion, dtype: float64
mismatch
0    0.725333
1    0.274667
Name: proportion, dtype: float64
mismatch
0    0.725333
1    0.274667
Name: proportion, dtype: float64


In [53]:
train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv", index=False)
test_df.to_csv("test.csv", index=False)

In [54]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

# Compute weights from training data
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["mismatch"]),
    y=train_df["mismatch"]
)

# Convert to torch tensor
class_weights = torch.tensor(
    weights,
    dtype=torch.float
)

print("Class Weights:", class_weights)

Class Weights: tensor([0.6894, 1.8201])


In [55]:
print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

print(train_df.columns.tolist())

(14000, 27)
(3000, 27)
(3000, 27)
['Ticket_ID', 'Ticket_Subject', 'Ticket_Description', 'Priority_Level', 'Ticket_Channel', 'Issue_Category', 'Resolution_Time_Hours', 'Customer_Email', 'clean_description', 'clean_text', 'rule_score', 'rule_evidence', 'priority_num', 'resolution_score', 'cluster', 'cluster_score', 'llm_score', 'severity_fusion', 'inferred_severity', 'assigned_priority_num', 'severity_delta', 'mismatch', 'mismatch_type', 'text', 'resolution_time_norm', 'channel_encoded', 'category_encoded']


# Model Training

In [56]:
import pandas as pd

train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("val.csv")
test_df = pd.read_csv("test.csv")

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(14000, 27)
(3000, 27)
(3000, 27)


In [57]:
import torch
import torch.nn as nn

from torch.utils.data import Dataset,DataLoader

from transformers import (AutoTokenizer, AutoModel, get_linear_schedule_with_warmup)

from peft import (get_peft_model, LoraConfig)

from sklearn.metrics import (accuracy_score,
    f1_score,
    classification_report,
    recall_score
)

device=torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [58]:
MODEL_NAME="distilbert-base-uncased"

MAX_LEN=128

BATCH_SIZE=32

EPOCHS= 5

tokenizer=AutoTokenizer.from_pretrained(
    MODEL_NAME
)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [59]:
class TicketDataset(Dataset):

    def __init__(self, df, tokenizer, max_len):

        self.texts = df["clean_text"].tolist()
        self.labels=df["mismatch"].tolist()
        self.channels=df["channel_encoded"].tolist()
        self.categories=df["category_encoded"].tolist()
        self.res_times = df["resolution_time_norm"].tolist()
        self.tokenizer=tokenizer
        self.max_len=max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self,idx):
        encoding=self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {

            "input_ids":
            encoding["input_ids"]
            .squeeze(0),

            "attention_mask":
            encoding["attention_mask"]
            .squeeze(0),

            "channel":
            torch.tensor(
                self.channels[idx],
                dtype=torch.float
            ),

            "category":
            torch.tensor(
                self.categories[idx],
                dtype=torch.float
            ),

            "res_time":
            torch.tensor(
                self.res_times[idx],
                dtype=torch.float
            ),


            "label":
            torch.tensor(
                self.labels[idx],
                dtype=torch.long
            )
        }

In [60]:
train_dataset=TicketDataset(train_df, tokenizer, MAX_LEN)
val_dataset=TicketDataset(val_df, tokenizer, MAX_LEN)
test_dataset=TicketDataset(test_df, tokenizer, MAX_LEN)
train_loader=DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader=DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader=DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [61]:
lora_config=LoraConfig(r=8, lora_alpha=16, lora_dropout=0.1, bias="none", target_modules=["q_lin", "v_lin"],

    inference_mode=False
)

In [62]:
class DistilBERTLoRAWithMetadata(
    nn.Module
):

    def __init__(
        self,
        model_name,
        lora_config,
        n_metadata=3,
        n_classes=2,
        dropout=0.3
    ):

        super().__init__()

        base_model=AutoModel.from_pretrained(
            model_name
        )

        self.encoder=get_peft_model(
            base_model,
            lora_config
        )

        hidden_size=(
            base_model
            .config
            .hidden_size
        )

        self.meta_proj=nn.Sequential(

            nn.Linear(
                n_metadata,
                32
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            )
        )

        self.classifier=nn.Sequential(

            nn.Linear(
                hidden_size+32,
                256
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                256,
                n_classes
            )
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        channel,
        category,
        res_time
    ):

        outputs=self.encoder(

            input_ids=input_ids,

            attention_mask=
            attention_mask
        )

        cls_output=(
            outputs
            .last_hidden_state[:,0,:]
        )

        meta=torch.stack(

            [
                channel,
                category,
                res_time
            ],

            dim=1
        )

        meta_out=self.meta_proj(
            meta
        )

        combined=torch.cat(

            [
                cls_output,
                meta_out
            ],

            dim=1
        )

        return self.classifier(
            combined
        )

In [63]:
import peft
import torchao

print(peft.__version__)
print(torchao.__version__)

0.19.1
0.17.0


In [64]:
model_clf=(
    DistilBERTLoRAWithMetadata(
        MODEL_NAME,
        lora_config
    )
).to(device)

print(model_clf)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

DistilBERTLoRAWithMetadata(
  (encoder): PeftModel(
    (base_model): LoraModel(
      (model): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-5): 6 x TransformerBlock(
              (attention): DistilBertSdpaAttention(
                (dropout): Dropout(p=0.1, inplace=False)
                (q_lin): lora.Linear(
                  (base_layer): Linear(in_features=768, out_features=768, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=768, out_features=8, bias=False)
      

In [65]:
class_weights = class_weights.to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

In [66]:
optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        model_clf.parameters()
    ),
    lr=2e-4,
    weight_decay=0.01
)

In [67]:
total_steps = len(train_loader) * EPOCHS

warmup_steps = int(
    0.1 * total_steps
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

In [68]:
def train_epoch(
    model,
    loader,
    optimizer,
    scheduler,
    criterion,
    device
):

    model.train()

    total_loss = 0

    all_preds = []

    all_labels = []

    for batch in loader:

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        channel = batch["channel"].to(device)

        category = batch["category"].to(device)

        res_time = batch["res_time"].to(device)

        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(
            input_ids,
            attention_mask,
            channel,
            category,
            res_time
        )

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        scheduler.step()

        total_loss += loss.item()

        all_preds.extend(
            torch.argmax(
                logits,
                dim=1
            ).cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

    return (

        total_loss/len(loader),

        accuracy_score(
            all_labels,
            all_preds
        ),

        f1_score(
            all_labels,
            all_preds,
            average="macro"
        )
    )

In [69]:
def eval_epoch(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    total_loss = 0

    all_preds = []

    all_labels = []

    with torch.no_grad():

        for batch in loader:

            input_ids = batch["input_ids"].to(device)

            attention_mask = batch["attention_mask"].to(device)

            channel = batch["channel"].to(device)

            category = batch["category"].to(device)

            res_time = batch["res_time"].to(device)

            labels = batch["label"].to(device)

            logits = model(
                input_ids,
                attention_mask,
                channel,
                category,
                res_time
            )

            loss = criterion(
                logits,
                labels
            )

            total_loss += loss.item()

            all_preds.extend(
                torch.argmax(
                    logits,
                    dim=1
                ).cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    return (

        total_loss / len(loader),

        accuracy_score(
            all_labels,
            all_preds
        ),

        f1_score(
            all_labels,
            all_preds,
            average="macro"
        ),

        all_preds,

        all_labels
    )

In [70]:
history = []

for epoch in range(
    1,
    EPOCHS + 1
):

    print(f"\n{'='*50}")
    print(f"EPOCH {epoch}/{EPOCHS}")
    print(f"{'='*50}")

    train_loss, train_acc, train_f1 = train_epoch(
        model_clf,
        train_loader,
        optimizer,
        scheduler,
        criterion,
        device
    )

    val_loss, val_acc, val_f1, val_preds, val_labels = eval_epoch(
        model_clf,
        val_loader,
        criterion,
        device
    )

    history.append({

        "epoch": epoch,

        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_f1": train_f1,

        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_f1": val_f1
    })

    print(
        f"\nTrain → Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Macro F1: {train_f1:.4f}"
    )

    print(
        f"Val   → Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | Macro F1: {val_f1:.4f}"
    )


EPOCH 1/5

Train → Loss: 0.6042 | Acc: 0.6702 | Macro F1: 0.6367
Val   → Loss: 0.5157 | Acc: 0.6990 | Macro F1: 0.6808

EPOCH 2/5

Train → Loss: 0.5366 | Acc: 0.6861 | Macro F1: 0.6670
Val   → Loss: 0.5020 | Acc: 0.7097 | Macro F1: 0.6931

EPOCH 3/5

Train → Loss: 0.5204 | Acc: 0.7034 | Macro F1: 0.6833
Val   → Loss: 0.4966 | Acc: 0.7163 | Macro F1: 0.6989

EPOCH 4/5

Train → Loss: 0.5133 | Acc: 0.7083 | Macro F1: 0.6880
Val   → Loss: 0.4918 | Acc: 0.7200 | Macro F1: 0.7026

EPOCH 5/5

Train → Loss: 0.5078 | Acc: 0.7115 | Macro F1: 0.6910
Val   → Loss: 0.4909 | Acc: 0.7273 | Macro F1: 0.7085


In [71]:
from sklearn.metrics import (
    classification_report,
    recall_score
)

print(
    "\n=== FINAL TEST EVALUATION ==="
)

test_loss,test_acc,test_f1,test_preds,test_labels=(
    eval_epoch(
        model_clf,
        test_loader,
        criterion,
        device
    )
)

print(
    f"\nTest Loss: {test_loss:.4f}"
)

print(
    f"Test Accuracy: {test_acc:.4f}"
)

print(
    f"Test Macro F1: {test_f1:.4f}"
)

print(
    "\n=== CLASSIFICATION REPORT ==="
)

print(
    classification_report(
        test_labels,
        test_preds,
        target_names=[
            "Consistent (0)",
            "Mismatch (1)"
        ],
        digits=4
    )
)

recalls=recall_score(
    test_labels,
    test_preds,
    average=None
)

print(
    "\n=== VERIFICATION THRESHOLD CHECK ==="
)

print(
    f"Accuracy ≥ 0.83: {'✅' if test_acc>=0.83 else '❌'} ({test_acc:.4f})"
)

print(
    f"Macro F1 ≥ 0.82: {'✅' if test_f1>=0.82 else '❌'} ({test_f1:.4f})"
)

print(
    f"Recall class 0 ≥ 0.78: {'✅' if recalls[0]>=0.78 else '❌'} ({recalls[0]:.4f})"
)

print(
    f"Recall class 1 ≥ 0.78: {'✅' if recalls[1]>=0.78 else '❌'} ({recalls[1]:.4f})"
)

results_df=pd.DataFrame({

    "Metric":[
        "Accuracy",
        "Macro_F1",
        "Recall_Class0",
        "Recall_Class1"
    ],

    "Value":[
        test_acc,
        test_f1,
        recalls[0],
        recalls[1]
    ]
})

print(
    "\n=== RESULTS DATAFRAME ==="
)

display(results_df)

history_df=pd.DataFrame(
    history
)

display(
    history_df
)


=== FINAL TEST EVALUATION ===

Test Loss: 0.5297
Test Accuracy: 0.7040
Test Macro F1: 0.6830

=== CLASSIFICATION REPORT ===
                precision    recall  f1-score   support

Consistent (0)     0.9035    0.6627    0.7646      2176
  Mismatch (1)     0.4772    0.8131    0.6014       824

      accuracy                         0.7040      3000
     macro avg     0.6904    0.7379    0.6830      3000
  weighted avg     0.7864    0.7040    0.7198      3000


=== VERIFICATION THRESHOLD CHECK ===
Accuracy ≥ 0.83: ❌ (0.7040)
Macro F1 ≥ 0.82: ❌ (0.6830)
Recall class 0 ≥ 0.78: ❌ (0.6627)
Recall class 1 ≥ 0.78: ✅ (0.8131)

=== RESULTS DATAFRAME ===


,Metric,Value
0,Accuracy,0.704000
1,Macro_F1,0.683009
2,Recall_Class0,0.662684
3,Recall_Class1,0.813107


,epoch,train_loss,train_acc,train_f1,val_loss,val_acc,val_f1
0,1,0.604234,0.670214,0.636651,0.515738,0.699000,0.680765
1,2,0.536558,0.686071,0.666969,0.501999,0.709667,0.693101
2,3,0.520428,0.703429,0.683294,0.496586,0.716333,0.698944
3,4,0.513337,0.708286,0.687998,0.491762,0.720000,0.702582
4,5,0.507846,0.711500,0.690963,0.490927,0.727333,0.708529


In [72]:
import os
import torch
import joblib

os.makedirs(
    "saved_model",
    exist_ok=True
)

torch.save(
    model_clf.state_dict(),
    "saved_model/model.pt"
)

tokenizer.save_pretrained(
    "saved_model/tokenizer"
)

joblib.dump(
    le_channel,
    "saved_model/channel_encoder.pkl"
)

joblib.dump(
    le_category,
    "saved_model/category_encoder.pkl"
)

joblib.dump(
    scaler,
    "saved_model/resolution_scaler.pkl"
)

results_df.to_csv(
    "saved_model/results.csv",
    index=False
)

history_df.to_csv(
    "saved_model/training_history.csv",
    index=False
)

print("Training Complete")
print("Model Saved")

Training Complete
Model Saved


# Dossier Generation using mistral

In [ ]:
import pandas as pd
import json
import torch

flagged = df[
    df["mismatch"] == 1
].copy()

hidden = flagged[
    flagged["mismatch_type"] == "Hidden Crisis"
].sample(
    min(
        25,
        len(
            flagged[
                flagged["mismatch_type"] == "Hidden Crisis"
            ]
        )
    ),
    random_state=42
)

false_alarm = flagged[
    flagged["mismatch_type"] == "False Alarm"
].sample(
    min(
        25,
        len(
            flagged[
                flagged["mismatch_type"] == "False Alarm"
            ]
        )
    ),
    random_state=42
)

dossier_sample = pd.concat(
    [hidden, false_alarm]
).reset_index(drop=True)

print(
    dossier_sample.shape
)

print(
    dossier_sample["mismatch_type"]
    .value_counts()
)

(50, 24)
mismatch_type
Hidden Crisis    25
False Alarm      25
Name: count, dtype: int64


In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)

MODEL = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

print("Mistral Loaded")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Mistral Loaded


In [ ]:
def build_prompt(row):

    return f"""
You are a support operations analyst.

Ticket Subject:
{row['Ticket_Subject']}

Ticket Description:
{row['Ticket_Description']}

Assigned Priority:
{row['Priority_Level']}

Inferred Severity:
{row['inferred_severity']}

Mismatch Type:
{row['mismatch_type']}

Keyword Evidence:
{row['rule_evidence']}

Resolution Time:
{row['Resolution_Time_Hours']} hours

Explain:

1. Why the assigned priority may not reflect the ticket urgency.
2. Which evidence supports the inferred severity.
3. Why the ticket is classified as {row['mismatch_type']}.

Use only the provided information.
Do not mention scores.
Do not invent information.
Maximum 3 sentences.
"""

In [ ]:
def generate_analysis(row):

    prompt = build_prompt(row)

    response = generator(
        prompt,
        max_new_tokens=120,
        do_sample=False,
        temperature=0.0
    )

    generated = response[0]["generated_text"]

    analysis = generated.replace(
        prompt,
        ""
    ).strip()

    return analysis

In [ ]:
results = []

for idx, row in dossier_sample.iterrows():

    try:

        analysis = generate_analysis(
            row
        )

    except Exception:

        analysis = (
            "Explanation generation failed."
        )

    keyword_evidence = (

        row["rule_evidence"]

        if pd.notna(
            row["rule_evidence"]
        )

        and str(
            row["rule_evidence"]
        ).strip() != ""

        else "No keyword evidence"
    )

    confidence = min(
        0.99,
        round(
            0.5 +
            abs(
                row["severity_delta"]
            ) * 0.15,
            2
        )
    )

    dossier = {

        "ticket_id":
        row["Ticket_ID"],

        "assigned_priority":
        row["Priority_Level"],

        "inferred_severity":
        int(
            row["inferred_severity"]
        ),

        "mismatch_type":
        row["mismatch_type"],

        "severity_delta":
        int(
            row["severity_delta"]
        ),

        "feature_evidence":[

            {
                "signal":
                "subject",

                "value":
                row["Ticket_Subject"]
            },

            {
                "signal":
                "keyword",

                "value":
                keyword_evidence
            },

            {
                "signal":
                "llm_severity",

                "value":
                str(
                    row["llm_score"]
                )
            },

            {
                "signal":
                "cluster_score",

                "value":
                str(
                    row["cluster_score"]
                )
            },

            {
                "signal":
                "resolution_time",

                "value":
                str(
                    row["Resolution_Time_Hours"]
                )
            }

        ],

        "constraint_analysis":
        analysis,

        "confidence":
        confidence
    }

    results.append(
        dossier
    )

    if len(results) % 20 == 0:

        print(
            f"Generated {len(results)} dossiers"
        )

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for ope

Generated 20 dossiers


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Settin

Generated 40 dossiers


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


In [ ]:
with open(
    "sample_dossiers.json",
    "w"
) as f:

    json.dump(
        results,
        f,
        indent=4
    )

print(
    "Saved sample_dossiers.json"
)

In [ ]:
for dossier in results[:3]:

    print(
        "=" * 100
    )

    print(
        json.dumps(
            dossier,
            indent=4
        )
    )

# Ablation study

Ablation 1: Generate Labels for Each Signal Individually

In [ ]:
ablation_results = {}

ablations = {

    "LLM Only":
    df["llm_score"],

    "Rule Only":
    df["rule_score"],

    "Resolution Only":
    df["resolution_score"],

    "Cluster Only":
    df["cluster_score"],

    "LLM + Resolution":
    0.7*df["llm_score"] +
    0.3*df["resolution_score"],

    "LLM + Rule":
    0.8*df["llm_score"] +
    0.2*df["rule_score"],

    "LLM + Resolution + Rule":
    0.6*df["llm_score"] +
    0.3*df["resolution_score"] +
    0.1*df["rule_score"],

    "All Signals":
    0.5*df["llm_score"] +
    0.25*df["resolution_score"] +
    0.15*df["rule_score"] +
    0.10*df["cluster_score"]
}

In [ ]:
def severity_from_score(score):

    severity = pd.Series(0,index=score.index)

    severity.loc[score>=0.75]=1
    severity.loc[score>=1.50]=2
    severity.loc[score>=2.25]=3

    return severity

In [ ]:
rows=[]

for name,score in ablations.items():

    inferred = severity_from_score(score)

    delta = (
        inferred
        - df["assigned_priority_num"]
    )

    mismatch = (
        abs(delta)>=1
    ).astype(int)

    rows.append({

        "Configuration":
        name,

        "Mismatch_Count":
        mismatch.sum(),

        "Mismatch_Rate":
        round(
            mismatch.mean(),
            4
        )
    })

ablation_df = pd.DataFrame(rows)

ablation_df

,Configuration,Mismatch_Count,Mismatch_Rate
0,LLM Only,11718,0.5859
1,Rule Only,12237,0.6118
2,Resolution Only,12430,0.6215
3,Cluster Only,17777,0.8888
4,LLM + Resolution,11718,0.5859
5,LLM + Rule,11718,0.5859
6,LLM + Resolution + Rule,12042,0.6021
7,All Signals,12387,0.6194
